In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
%load_ext snakeviz

import sys
import os
import logging
import matplotlib
# a bug in jupyter / ipympl / matplotlib needs this here when using %maptlotlib widget
# somehow rc_context is broken in that case
matplotlib.rc('text.latex', preamble=r"\usepackage{siunitx}\usepackage{xfrac}")
# to adjust figure uncomment here and comment out %matplotlib widget, restart kernel and 
# make sure jupyterlab is running in a x-forwarded and connected ssh session and the right display is set
# os.environ['DISPLAY'] = 'localhost:17.0'
# matplotlib.use("qtagg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import scipy.stats.qmc

import ray
from polyuq import *
from polyuq.data_manager import DataManager
from helpers import get_pcd

%aimport -sys -logging -matplotlib -matplotlib.pyplot -numpy -pandas - scipy.stats -scipy.stats.qmc -ray

### Verification example: numerical modal analysis ###

System: Vibrating beam, hinged and flexibly supported
Input parameters and uncertainties:
- Cross-sectional area properties: polymorphic - e.g. due to manufacturing tolerances 
    - ellipse radius $a=0.9$: deterministic (to reduce dimension of design space)
    - ellipse radius $b$: imprecision
        - [0.85 ... 0.95] m deformed state; m=1
    - Wall thickness $t$: mixed - imprecision & variability
        - [5.9,6.1e-3] nominal; m=0.8
        - Normal(6e-3, 1e-4); m= 0.2

- Additional mass (plattforms, cables, antennas, etc.): imprecision
    - [0.2-1 kN/m Petersen: Stahlbau]

- Pre-tension of Guy wires: incompleteness -  due to unknown nominal state and variable temperature effects
    - Nominal state $\mu_N$ : mixed - deterministic & imprecision
        - 60000 N design value; m=0.75
        - [40000 - 180000] N; m=0.25
    - Normal($\mu_N$,2655) N; 
        -Standard deviation obtained from Geyer Monitoring Temperature Data with $N_\text{wire} = N_\text{nominal} - \alpha_t \Delta_t E A \cos^2(\alpha)$

- Cross-Sectional Area of wires: imprecision - due to manufacturing tolerances
    - [0.0007,0.0008] m; m=1

- Damping ratio: imprecision
    - [Petersen, 2017, P143] for low, medium, high vibration amplitudes: 
        - $\Lambda$=[0.008, 0.013, 0.018] (steel) + [0.035,0.04,0.06] (guyed masts) + [0.004,0.005,0.006] (concrete foundation)
        - $\zeta = \frac{\Lambda}{\sqrt{(2\pi)^2+\Lambda^2}} \approx \frac{\Lambda}{2\pi}$
    - VDI2038-2, Table 1, p. 14: $\zeta$=[0.0016, 0.015]
    - damping ratio [each m=0.3]: [0.0075, 0.0092],[0.0075, 0.0134], [0.0016, 0.015] 

- Viscosity of the TMD damper: incompleteness - due to unknown temperature-viscosity relations
    - Standard Deviation $\sigma_D$:
    - mean temp = 8.62, stdd temp = 8.19 (Geyer north sensor)
    - $F = \eta A   \frac{v}{h}$ ($\eta$: Viscosity, $A$ shear area, $v$ fluid velocity, $h$ height of the fluidlayer)
    - Daempfungskonstante: $\frac{F}{v} = \eta \frac{A}{h}$ [N s m^-2  m^2 m^-1 = N s m^-1]
    - $\eta = -6.23e-3 T + 8.45e-1$
    - $dD(T) = -1.557 T + 2.11e2$
    - $\sigma_{dD} = 12.75$
        - (10,15) N s m^-1; m=0.8
        - (5,20) N s m^-1; m=0.2
    - Normal(197.61, $\sigma_{dD}$)  N s m^-1
    
- Icing: incompletenes & variability & imprecision
     - probability of occurence: mixed - deterministic & imprecision 
         - [[28.2/365 = 0.077], [1/365, 77/365]] https://www.dwd.de/DE/wetter/wetterundklima_vorort/thueringen/erfurt/_node.html
    - occurence: variability
        - Bernoulli distribution
    - mass: imprecision
        - (EN DIN EN 1991-1-3/NA:2019-04: [0.5 - 1] kN/m )

- Young's Modulus $E$: deterministic 2.1e11 N/m^2
- mass density $\rho$: deterministic 7850 kg/m^3
- TMD mass $m$: deterministic 800 kg
- TMD stiffness $k_{y,z}$: deterministic 1025.48 N/m
- Degrees-of-freedom $n$: deterministic - 200
- number of modes considered $n_m$: deterministic - 14
- Sampling rate $f_s$: deterministic - 10 Hz
- Frequency lines $N$: deterministic - 1025
- Measurement locations $x_m$: deterministic - 5


    
Output parameters:
- damped modal frequencies: $f_{d,1} \ldots f_{d,14}$
- modal damping: $\zeta_1 \ldots \zeta_{14}$
- FRF (Magnitude) at length $x$ (also includes modeshapes) due to tip excitation: $\mathcal{H}_\mathrm{a}(\omega_\mathrm{f})$

In [ ]:
from examples.UQ_Modal_FEM import mapping_function, vars_definition, est_imp, opt_inc

##### Uncertainty Modeling

In [ ]:
vars_ale, vars_epi, arg_vars = vars_definition()
dim_ex = 'cartesian'

# %%snakeviz
N_mcs_ale = 13717 # N_mcs = 1e6 = N_mcs_ale * N_mcs_epi
N_mcs_epi = 729 # = 3^6 = 2.56^n_imp ~ 3^n_imp -> cover every corner and midpoints in a full-factorial design (but distributed)
use_dm = True
result_dir = 'polyuq_results/'

labels = {'damp_freqs':[['$f_{1,x\\text{-}z}^\mathrm{l}$','$f_{1,x\\text{-}y}^\mathrm{l}$'],
                        ['$f_{1,x\\text{-}z}^\mathrm{u}$','$f_{1,x\\text{-}y}^\mathrm{u}$'],
                        ['$f_{0,x\\text{-}z}$','$f_{0,x\\text{-}y}$'],
                        ['$f_{2,x\\text{-}z}$','$f_{2,x\\text{-}y}$'],
                        ['$f_{3,x\\text{-}z}$','$f_{3,x\\text{-}y}$'],
                        ['$f_{4,x\\text{-}z}$','$f_{4,x\\text{-}y}$'],
                        ['$f_{5,x\\text{-}z}$','$f_{5,x\\text{-}y}$']],
          'zetas':     [['$\zeta_{1,x\\text{-}z}^\mathrm{l}$','$\zeta_{1,x\\text{-}y}^\mathrm{l}$'],
                        ['$\zeta_{1,x\\text{-}z}^\mathrm{u}$','$\zeta_{1,x\\text{-}y}^\mathrm{u}$'],
                        ['$\zeta_{0,x\\text{-}z}$','$\zeta_{0,x\\text{-}y}$'],
                        ['$\zeta_{2,x\\text{-}z}$','$\zeta_{2,x\\text{-}y}$'],
                        ['$\zeta_{3,x\\text{-}z}$','$\zeta_{3,x\\text{-}y}$'],
                        ['$\zeta_{4,x\\text{-}z}$','$\zeta_{4,x\\text{-}y}$'],
                        ['$\zeta_{5,x\\text{-}z}$','$\zeta_{5,x\\text{-}y}$']]
         }

### Sampling

In [ ]:
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
if True:
    
    poly_uq.load_state(os.path.join(result_dir,'polyuq_samp.npz'), differential='samp')
    display(poly_uq.var_supp)
    # seed 1165763483 scrambles adequately to achieve good discrepancy values even for few (100) samples
    poly_uq.sample_qmc(10, 10, check_sample_sizes=False, seed=1165763483)
    display(poly_uq.var_supp)
    # poly_uq.save_state(os.path.join(result_dir,'polyuq_samp.npz'), differential='samp')
else:
    poly_uq.load_state(os.path.join(result_dir,'polyuq_samp.npz'), differential='samp')

if False:
    import seaborn as sns
    sns.pairplot(poly_uq.inp_samp_prim, kind='hist')
dm_grid = None

### Propagation

In [ ]:
'''
blimits -l Batch72 
CPULIMIT                
4320.0 min of makalu101 -> 9 hrs @ 8 tasks -> 24 hrs @ 3 tasks -> 72 hrs @ 1 task
RUNLIMIT                
4800.0 min
TASKLIMIT               
512

FILELIMIT STACKLIMIT CORELIMIT MEMLIMIT PROCESSLIMIT
125500000 K  512000 K  100000 K  380000000 K    1000      
 
blimits -l Batch24 
CPULIMIT                
1440.0 min of makalu101 -> 6 hrs @ 4 tasks -> 24 hrs @ 1 task
TASKLIMIT               
512

FILELIMIT STACKLIMIT CORELIMIT MEMLIMIT PROCESSLIMIT
100000000 K  512000 K  100000 K  80000000 K     100   
 

blimits -l BatchXL 
TASKLIMIT               
4000

CORELIMIT MEMLIMIT PROCESSLIMIT THREADLIMIT
100000 K  256000000 K   16384        32768  

'''

if dm_grid is None:
    dm_grid = DataManager.from_existing('uq_modal_beam.nc',
                                        result_dir=os.path.join(result_dir, 'samples'), 
                                        working_dir='polyuq_results/')

# with dm_grid.get_database('out',rw=True) as out_ds:
#     out_ds['_runtimes'][:]=np.nan
    
# dm_grid.evaluate_samples(mapping_function, arg_vars, 
#                      ret_names={'damp_freqs':('modes',), 'zetas':('modes',), 'frf':('frequencies','space',)}, 
#                      default_len={'modes':14, 'frequencies':1025, 'space':2}, dry_run=False,
#                     chunks_submit=100000, chunks_save=15000, scramble_evaluation=False, re_eval_sample='1ddf809fb7f3_b5b3e888fca2')
eval_cost = 22 # seconds
num_workers = 440
sav_time= 800 # seconds
chunks_save = sav_time/eval_cost*num_workers # 16000 
todo = False
while todo:
    logger = logging.getLogger('polyuq.data_manager')
    logger.setLevel(level=logging.INFO)
    logger = logging.getLogger('model.mechanical')
    logger.setLevel(level=logging.WARNING)
    
    todo = dm_grid.evaluate_samples(mapping_function, arg_vars, 
                         ret_names={'damp_freqs':('modes',), 'zetas':('modes',), 'frf':('frequencies','space',)}, default_len={'modes':14, 'frequencies':1025, 'space':3}, 
                         use_lock=False, dry_run=False,
                         chunks_submit=250000, chunks_save=10000, scramble_evaluation=False)

In [ ]:
ray.shutdown()

### export from datamanager

In [ ]:
if dm_grid is None:
    dm_grid = DataManager.from_existing('uq_modal_beam.nc',
                                        result_dir=os.path.join(result_dir, 'samples'), 
                                        working_dir='polyuq_results/')
with dm_grid.get_database('out',False) as out_ds:
    out_ds_keep = out_ds.load().copy()
out_ds = out_ds_keep

In [ ]:
def export(poly_uq, ret_name, ret_ind, out_ds, result_dir):
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    samp_path = os.path.join(result_dir,'polyuq_samp.npz')
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
    imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')

    poly_uq.load_state(samp_path, differential='samp')
    poly_uq.N_mcs_ale = 13717
    
    if os.path.exists(prop_path):
        poly_uq.load_state(prop_path, differential='prop')
    if os.path.exists(imp_path):
        poly_uq.load_state(imp_path, differential='imp')
    
    poly_uq.from_data_manager(None, ret_name, ret_ind, out_ds)
    
    if ret_name == 'damp_freqs':
        poly_uq.out_valid = [np.nanmin(poly_uq.out_samp),np.nanmax(poly_uq.out_samp)]
    poly_uq.save_state(prop_path, differential='prop')
    if os.path.exists(imp_path):
        poly_uq.save_state(imp_path, differential='imp')
        
        
if True:
    for ret_name in ['damp_freqs','zetas','frf'][2:]:
        if ret_name == 'frf':
            # continue
            inds = range(534*3,1025*3)
        else:
            continue
            inds = range(14)

        for ind in inds:
            if ret_name == 'frf':
                ret_ind = {'frequencies':ind//3, 'space':ind%3}
                if ind%3!=2:
                    continue
            else:
                ret_ind = {'modes':ind}
            try:
                export(PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex), ret_name, ret_ind, out_ds, result_dir)
            except Exception as e:
                print(e)
                yesno = input('retry? y/n')
                if yesno=='y':
                    export(PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex), ret_name, ret_ind, out_ds, result_dir)
                
                

else:
    export(PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex), ret_name, ret_ind, out_ds, result_dir)

### define output quantity

In [ ]:


logger= logging.getLogger('polyuq.polymorphic_uncertainty')
logger.setLevel(level=logging.INFO)

ret_name = ['damp_freqs','zetas','frf'][0]
if ret_name == 'frf':
    ret_ind = {'frequencies':105, 'space':2}
else:
    ret_ind = {'modes':7}
ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
samp_path = os.path.join(result_dir,'polyuq_samp.npz')
prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')

### Estimate imprecision

In [ ]:
# ret_ind = {'frequencies':533, 'space':1}
# ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'

poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

est_imp(poly_uq, result_dir, ret_name, ret_ind)


In [ ]:
plt.close('all')
for ret_name in ['damp_freqs','zetas','frf']:
    
    fig1, ax1 = plt.subplots()
    if ret_name == 'frf':
        # continue
        inds = range(10*3)
    else:
        continue
        inds = range(14)
    
    all_undershots = []
    num_undershots = []
    all_exceeds = []
    num_exceeds = []
    for ind in inds:
        if ret_name == 'frf':
            ret_ind = {'frequencies':ind//3, 'space':ind%3}
            if ind%3==0:
                continue
        else:
            ret_ind = {'modes':ind}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
        samp_path = os.path.join(result_dir,'polyuq_samp.npz')
        prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
        imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
        
        if not os.path.exists(imp_path): continue    
    
        poly_uq.load_state(samp_path, differential='samp')
        poly_uq.load_state(prop_path, differential='prop')
        poly_uq.load_state(imp_path, differential='imp')
        
        if poly_uq.imp_foc is not None:
            samp_fin = np.nonzero(
                    np.any(
                        np.isnan(poly_uq.imp_foc[:,:,0]),
                        axis=1)
                )[0]
            print(samp_fin)
            if len(samp_fin)>0:
                start_ale = np.min(samp_fin)
            else:
                start_ale = poly_uq.imp_foc.shape[0]
            print(start_ale)
        
        undershot = poly_uq.intp_undershot
        all_undershots.append(undershot[1] / undershot[0] / (poly_uq.out_valid[1]-poly_uq.out_valid[0])*100)
        all_undershots.append(0)
        num_undershots.append(undershot[0])
        num_undershots.append(poly_uq.N_mcs_ale*len(poly_uq.imp_hyc_foc_inds) - undershot[0])
        
        exceed = poly_uq.intp_exceed
        all_exceeds.append(exceed[1] / exceed[0] / (poly_uq.out_valid[1]-poly_uq.out_valid[0])*100)
        all_exceeds.append(0)
        num_exceeds.append(exceed[0])
        num_exceeds.append(poly_uq.N_mcs_ale*len(poly_uq.imp_hyc_foc_inds) - exceed[0])
        
        ax1.hist(100-poly_uq.intp_errors*100, bins=np.arange(101), alpha=0.1)
        
    ax1.set_title(f'Interpolator LOO Cross-Validation {ret_name}')
    ax1.set_xlabel("Accuracy [\%]")
    ax1.set_ylabel("Sample Count [-]")
    fig1.savefig(f'figures/imp_loo_accuracy{ret_name}.png')
    fig1.savefig(f'figures/imp_loo_accuracy{ret_name}.pdf')

    plt.figure()
    plt.hist(all_undershots, weights= num_undershots, bins=50)
    plt.title(ret_name + ' undershot error')

    plt.figure()
    plt.hist(all_exceeds, weights= num_exceeds, bins=50)
    plt.title(ret_name + ' exceed error') 

### Estimate Incompleteness

In [ ]:
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
result_dir = 'polyuq_results/'
ret_name = 'frf'
for freq_ind in range(380,1025):
    for space_ind in [1,2]:
        ret_ind = {'frequencies':freq_ind, 'space':space_ind}
        opt_inc(poly_uq, result_dir, ret_name, ret_ind)

### Plots

In [ ]:
plt.close('all')
with matplotlib.rc_context(get_pcd('BB15')):
    ret_name = 'damp_freqs'
    for mode_ind in range(14):
        plt.figure()
        ret_ind = {'modes':mode_ind}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'

        arr = np.load(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz'))
        focals_stats, focals_mass = arr['self.focals_stats'], arr['self.focals_mass']
        if ret_name == 'zetas':
            focals_stats *= 100
        plot_focals(focals_stats[0,:,:], focals_mass, plt.gca())
        if ret_name == 'damp_freqs':
            # plt.xlabel(f'Frequency {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
            plt.xlabel(f'Eigenfrequenz {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
        elif ret_name == 'zetas':
            # plt.xlabel(f'Damping {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
            plt.xlabel(f'Dämpfung {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')

        plt.ylim((0,1))
        # plt.ylabel('Cumulative Mass [-]')
        plt.ylabel('Kumulative Evidenz [-]')
        plt.subplots_adjust(top=0.97, bottom=0.135, left=0.105, right=0.97, wspace=0.045)
        
        # plt.savefig(f'figures/inc_avg_foc-{ret_dir}.png')
        # plt.savefig(f'figures/inc_avg_foc-{ret_dir}.pdf')
        plt.savefig(f'figures/inc_avg_foc-{ret_dir}.png')
        plt.savefig(f'figures/inc_avg_foc-{ret_dir}.pdf')
        
        plt.show()

In [ ]:
with matplotlib.rc_context(get_pcd('BB15')):
    fig, axes = plt.subplots(1,3, sharey=True)
    ret_name='damp_freqs'
    colors=['dimgrey','grey']
    linestyles = ['solid', (0, (5, 1))]
    add_mode = 0 # 0 or 4
    
    if not add_mode:
        axes = np.insert(axes, 0, axes[0])
    for mode_pair,ax in enumerate(axes):
        for pair_member in range(2):
            mode_ind = (mode_pair + add_mode)* 2 + pair_member
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            arr = np.load(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz'))
            focals_stats, focals_mass = arr['self.focals_stats'], arr['self.focals_mass']

            bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)

            dbins = np.diff(bins_bel)
            pl_stats[0,1:] /= dbins
            if not add_mode and mode_pair < 2:
                pl_stats /= 2
            # print(pl_stats.shape, bins_bel.shape)
            # bins_bel = np.insert(bins_bel, 0, bins_bel[0], axis=0)
            bins_bel = np.append(bins_bel, bins_bel[-1:], axis=0)
            # pl_stats = np.insert(pl_stats, 0, 0, axis=1)
            pl_stats = np.append(pl_stats, [[0]], axis=1)

            ax.step(bins_bel, pl_stats[0,:], where='post',label=labels[ret_name][mode_pair+add_mode][pair_member], color=colors[pair_member], linestyle=linestyles[pair_member])
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.legend(loc='upper left')
        
    # custom_lines = [matplotlib.lines.Line2D([0], [0], color=colors[i], ls=linestyles[i]) for i in range(2)]
    # fig.legend(custom_lines, ['$x$-$z$-plane', '$x$-$y$-plane',], loc=(0.78,0.84))

    # if not add_mode:
    #     axes[0].set_xlabel('$f^\\text{u,l}_{1} [\si{{\hertz}}]$')
    #     axes[2].set_xlabel('$f_{2} [\si{{\hertz}}]$')
    #     axes[3].set_xlabel('$f_{3} [\si{{\hertz}}]$')
    # else:
    #     axes[0].set_xlabel('$f_{4} [\si{{\hertz}}]$')
    #     axes[1].set_xlabel('$f_{5} [\si{{\hertz}}]$')
    #     axes[2].set_xlabel('$f_{6} [\si{{\hertz}}]$')
    if not add_mode:
        # axes[0].set_xlabel('Frequency $[\si{{\hertz}}]$')
        # axes[2].set_xlabel('Frequency $[\si{{\hertz}}]$')
        # axes[3].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[0].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')
        axes[2].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')
        axes[3].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')
    else:
        # axes[0].set_xlabel('Frequency $[\si{{\hertz}}]$')
        # axes[1].set_xlabel('Frequency $[\si{{\hertz}}]$')
        # axes[2].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[0].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')
        axes[1].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')
        axes[2].set_xlabel('Eigenfrequenz $[\si{{\hertz}}]$')

    axes[0].set_ylim(ymin=0)
    # axes[0].set_ylabel('Mass Density [$\sfrac{1}{\si{\hertz}}$]')
    axes[0].set_ylabel('kum. Evidenzdichte [$\sfrac{1}{\si{\hertz}}$]')
    plt.subplots_adjust(top=0.99, bottom=0.22, left=0.115, right=0.97, wspace=0.12)
    if not add_mode:
        # plt.savefig(f'figures/inc_avg_pl-damp_freqs-low.png')
        # plt.savefig(f'figures/fi'gures/examples/uq_modal_beam/inc_avg_pl-damp_freqs-low.pdf')
        plt.savefig(f'figures/inc_avg_pl-damp_freqs-low.png')
        plt.savefig(f'figures/inc_avg_pl-damp_freqs-low.pdf')
    else:
        # plt.savefig(f'figures/inc_avg_pl-damp_freqs-high.png')
        # plt.savefig(f'figures/inc_avg_pl-damp_freqs-high.pdf')
        plt.savefig(f'figures/inc_avg_pl-damp_freqs-high.png')
        plt.savefig(f'figures/inc_avg_pl-damp_freqs-high.pdf')
    plt.show()
        

In [ ]:
with matplotlib.rc_context(get_pcd('BB15')):
    fig, ax = plt.subplots(1,1)
    ret_name='zetas'
    add_mode = 0
    # add_mode = 0
    if not add_mode:
        colors=['dimgrey','grey']
    else:
        colors=['dimgrey','grey', 'lightgrey']
    linestyles = ['solid', (0, (5, 1))]


    for mode_pair, color in enumerate(colors):
        for pair_member in range(2):
            mode_ind = (mode_pair  + add_mode) * 2 + pair_member
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            arr = np.load(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz'))
            focals_stats, focals_mass = arr['self.focals_stats'], arr['self.focals_mass']

            bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
            bins_bel *= 100
            dbins = np.diff(bins_bel)
            pl_stats[0,1:] /= dbins
            if len(colors)==2:
                pl_stats /= 2
            bins_bel = np.append(bins_bel, bins_bel[-1:], axis=0)
            pl_stats = np.append(pl_stats, [[0]], axis=1)

            ax.step(bins_bel, pl_stats[0,:], where='post', color=color, linestyle=linestyles[pair_member], label=labels[ret_name][mode_pair+add_mode][pair_member])
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    # ax.set_xlabel(f'$f_{{{mode_pair+add_mode+1},y,z}} [\si{{\hertz}}]$')

    ax.set_ylim(ymin=0)
    ax.set_xlim(xmin=0)
    # ax.set_ylabel('Mass Density [$\sfrac{1}{\si{\percent}}$]')
    # ax.set_xlabel(f'Damping $\zeta [\si{{\percent}}]$')
    ax.set_ylabel('kumulative Evidenzdichte [$\sfrac{1}{\si{\percent}}$]')
    ax.set_xlabel(f'Dämpfung $\zeta [\si{{\percent}}]$')

    plt.legend(loc='upper right')
    plt.subplots_adjust(top=0.97, bottom=0.135, left=0.105, right=0.97, wspace=0.045)

    # if not add_mode:
    #     # plt.savefig(f'figures/inc_avg_pl-zetas-low.png')
    #     # plt.savefig(f'figures/inc_avg_pl-zetas-low.pdf')
    #     plt.savefig(f'figures/inc_avg_pl-zetas-low.png')
    #     plt.savefig(f'figures/inc_avg_pl-zetas-low.pdf')
    # else:
    #     # plt.savefig(f'figures/inc_avg_pl-zetas-high.png')
    #     # plt.savefig(f'figures/inc_avg_pl-zetas-high.pdf')
    #     plt.savefig(f'figures/inc_avg_pl-zetas-high.png')
    #     plt.savefig(f'figures/inc_avg_pl-zetas-high.pdf')
    plt.show()

In [ ]:
plt.close('all')
ret_name='damp_freqs'
for mode in range(14):
    
    ret_ind = {'modes':mode}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
    samp_path = os.path.join(result_dir,'polyuq_samp.npz')
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
    imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
    
    for inc_path_part in ['polyuq_hist_inc.npz', 'polyuq_cdf_inc.npz']:

        # inc_path_part = 'polyuq_hist_inc.npz'
        # inc_path_part = 'polyuq_cdf_inc.npz'
        inc_path = os.path.join(result_dir, 'estimations', ret_dir, inc_path_part)
        poly_uq.load_state(samp_path, differential='samp')
        poly_uq.load_state(prop_path, differential='prop')
        poly_uq.load_state(imp_path, differential='imp')
        poly_uq.load_state(inc_path, differential='inc')
        focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
        n_stat = focals_stats.shape[0]
        if False:
            fix, axes = plt.subplots(8,5,sharex=True, sharey=True)
            for i_stat in range(n_stat):
                plot_focals(focals_stats[i_stat,:,:],focals_mass, axes.flat[i_stat])

        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
        n_bins_bel = bins_bel.shape[0]
        with matplotlib.rc_context(get_pcd('BB15')):
            # fig, ax1 = plt.subplots()
            # plt.rc('text',usetex=True)
            # print(matplotlib.rcParams)
            if 'cdf' in inc_path_part:
                plt.figure()
                n_stat,n_hyc,_ = focals_stats.shape
                target_probabilities = np.linspace(0,1,n_stat)
                for i_hyc in range(n_hyc):
                    for minmax in range(2):
                        plt.step(focals_stats[:,i_hyc,minmax],target_probabilities,  where='post', c='grey')

                if ret_name == 'zetas':
                    # labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]
                    # plt.xlabel(f'Damping {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                    # plt.ylabel('Cumulative Probability [$-$]')
                    plt.xlabel(f'Dämpfung {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                    plt.ylabel('Kumulative Wahrscheinlichkeit [$-$]')
                elif ret_name == 'damp_freqs':
                    # plt.xlabel(f'Frequency {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                    # plt.ylabel('Cumulative Probability [$-$]')
                    plt.xlabel(f'Eigenfrequenz {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                    plt.ylabel('Kumulative Wahrscheinlichkeit [$-$]')

                plt.ylim((0,1))
                plt.xlim((np.nanmin(focals_stats), np.nanmax(focals_stats))) 
                plt.subplots_adjust(top=0.975,bottom=0.145, left=0.09,right=0.98)
                # plt.savefig(f'figures/inc_cdf_foc-{ret_dir}.png')
                # plt.savefig(f'figures/inc_cdf_foc-{ret_dir}.pdf')
                plt.savefig(f'figures/inc_cdf_foc-{ret_dir}.png')
                plt.savefig(f'figures/inc_cdf_foc-{ret_dir}.pdf')
                
            if True:
                plt.figure()
                if 'hist' in inc_path_part:
                    nbin_fact=20
                    n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
                    bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(poly_uq.N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2)
                    if ret_name == 'zetas':
                        bins_densities *=100

                    # db = np.array(np.diff(bins_densities), float)
                    # for i_hyc in range(focals_stats.shape[1]):
                    #     for mi_ma in range(2):
                    #         focals_stats[:,i_hyc,mi_ma] /= db#*np.sum(focals_stats[:,i_hyc,mi_ma])
                    # bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
                    # n_bins_bel = bins_bel.shape[0]

                    plt.pcolormesh(bins_densities[:-1], bins_bel, pl_stats.T, cmap='Greys')#, shading='gouraud')
                    cbar = plt.colorbar()
                    # cbar.set_label('Plausibility [-]')
                    cbar.set_label('Plausibilität [-]')
                    if ret_name == 'zetas':
                        # plt.xlabel(f'Damping {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                        # plt.ylabel('Weighted Count [$\sfrac{1}{\si{\percent}_\\mathrm{{crit.}}}$]')
                        plt.xlabel(f'Dämpfung {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                        plt.ylabel('Gewichtete Anzahl [$\sfrac{1}{\si{\percent}_\\mathrm{{crit.}}}$]')
                    elif ret_name == 'damp_freqs':

                        # plt.xlabel(f'Frequency {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                        # plt.ylabel('Weighted Count [\\si{\\per\\hertz}]')
                        plt.xlabel(f'Eigenfrequenz {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                        plt.ylabel('Gewichtete Anzahl [\\si{\\per\\hertz}]')
                    plt.ylim((bins_bel.min(), bins_bel.max()))
                    plt.xlim((bins_densities.min(), bins_densities.max()))
                    plt.subplots_adjust(top=0.975,bottom=0.145, left=0.1,right=1.0)
                    # plt.savefig(f'figures/inc_hist_pl-{ret_dir}.png')
                    # plt.savefig(f'figures/inc_hist_pl-{ret_dir}.pdf')
                    plt.savefig(f'figures/inc_hist_pl-{ret_dir}.png')
                    plt.savefig(f'figures/inc_hist_pl-{ret_dir}.pdf')

                elif 'cdf' in inc_path_part:
                    n_stat = pl_stats.shape[0]
                    target_probabilities = np.linspace(0,1,n_stat)
                    if ret_name == 'zetas':
                        bins_bel *= 100
                    plt.pcolormesh(bins_bel, target_probabilities, pl_stats, cmap='Greys')#, shading='gouraud')
                    cbar = plt.colorbar()
                    # cbar.set_label('Plausibility [-]')
                    cbar.set_label('kumulative Evidenz [-]')
                    if ret_name == 'zetas':
                        # plt.xlabel(f'Damping {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                        # plt.ylabel('Cumulative Probability [$-$]')
                        plt.xlabel(f'Dämpfung {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
                        plt.ylabel('Kumulative Wahrscheinlichkeit [$-$]')
                    elif ret_name == 'damp_freqs':
                        # plt.xlabel(f'Frequency {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                        # plt.ylabel('Cumulative Probability [$-$]')
                        plt.xlabel(f'Eigenfrequenz {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\hertz}}]$')
                        plt.ylabel('Kumulative Wahrscheinlichkeit [$-$]')
                    plt.ylim((0,1))
                    plt.xlim((bins_bel.min(), bins_bel.max()))
                    plt.subplots_adjust(top=0.975,bottom=0.145, left=0.09,right=1.0)
                    # plt.savefig(f'figures/inc_cdf_pl-{ret_dir}.png')
                    # plt.savefig(f'figures/inc_cdf_pl-{ret_dir}.pdf')
                    plt.savefig(f'figures/inc_cdf_pl-{ret_dir}.png')
                    plt.savefig(f'figures/inc_cdf_pl-{ret_dir}.pdf')
            #plt.show()
            # break

In [ ]:
plt.close('all')
ret_name='zetas'

with matplotlib.rc_context(get_pcd('BB15')):
    fig, axes = plt.subplots(1,4, sharex=True, sharey=True)
    for mode in range(4):

        ret_ind = {'modes':mode}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
        poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
        samp_path = os.path.join(result_dir,'polyuq_samp.npz')
        prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
        imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
        inc_path_part = 'polyuq_cdf_inc.npz'
        inc_path = os.path.join(result_dir, 'estimations', ret_dir, inc_path_part)
        
        poly_uq.load_state(samp_path, differential='samp')
        poly_uq.load_state(prop_path, differential='prop')
        poly_uq.load_state(imp_path, differential='imp')
        poly_uq.load_state(inc_path, differential='inc')
        focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
        n_stat = focals_stats.shape[0]

        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
        n_bins_bel = bins_bel.shape[0]

        n_stat = pl_stats.shape[0]
        target_probabilities = np.linspace(0,1,n_stat)
        if ret_name == 'zetas':
            bins_bel *= 100
            
        plt.sca(axes[mode])
        mappable = plt.pcolormesh(bins_bel, target_probabilities, pl_stats, cmap='Greys')#, norm=matplotlib.colors.PowerNorm(gamma=3))#, shading='gouraud')
        if mode==3:
            cbar = fig.colorbar(mappable)
            # cbar.set_label('Plausibility [-]')
            cbar.set_label('kumulative Evidenz [-]')
        plt.ylim((0,1))
        # print(labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2])
        plt.title(labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2], fontsize=9)
        # plt.legend(title = labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2])
        # plt.xlim((bins_bel.min(), bins_bel.max()))
        fig.supxlabel(f'Dämpfung  $\zeta_1 [\si{{\percent}}_\\mathrm{{crit.}}]$', fontsize=9)
        axes[0].set_ylabel('kumulative Wahrsch. [$-$]')
        plt.subplots_adjust(top=0.94,bottom=0.12, left=0.1,right=0.9, wspace=0.075)
        plt.savefig(f'figures/inc_cdf_pl-zetas-low.png', dpi=300)
        plt.savefig(f'figures/inc_cdf_pl-zetas-low.pdf')


In [ ]:
fig.supxlabel(f'Dämpfung {labels[ret_name][ret_ind["modes"]//2][ret_ind["modes"]%2]} $[\si{{\percent}}_\\mathrm{{crit.}}]$')
axes[0].set_ylabel('Kumul. Wahrsch. [$-$]')
plt.subplots_adjust(top=0.965,bottom=0.3, left=0.2,right=0.9, wspace=0.075)

In [ ]:
ret_name = 'frf'
all_focals = []
space_ind = 2
for freq_ind in range(1025):
    ret_ind = {'frequencies':freq_ind, 'space':space_ind}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'

    inc_path = os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz')
    arr = np.load(inc_path)
    focals_stats, focals_mass = arr['self.focals_stats'], arr['self.focals_mass']
    all_focals.append(focals_stats)
    # print(focals_stats)
all_focals_this = np.stack([foc[0,:,:] for foc in all_focals])
np.savez(os.path.join(result_dir,'estimations',f'frf-all.{space_ind}.npz'), all_focals_this=all_focals_this, focals_mass=focals_mass)

In [ ]:
space_ind = 1
freq_ind = 1025-1

arr = np.load(os.path.join(result_dir,'estimations',f'frf-all.{space_ind}.npz'))
all_focals_this, focals_mass = arr['all_focals_this'][:freq_ind,:], arr['focals_mass']

all_focals_this *= 1000
# all_focals_this = np.log10(all_focals_this)
bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(all_focals_this,focals_mass,100)
n_bins_bel = bel_stats.shape[1]

print(np.min(bins_bel), np.max(bins_bel), np.min(all_focals_this), np.max(all_focals_this), len(bins_bel))
cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
with matplotlib.rc_context(get_pcd('BB15')):

    plt.figure()
    im = plt.imshow(pl_stats.T, aspect='auto', extent= (0, 5/1025*freq_ind, np.min(all_focals_this), np.max(all_focals_this)),
                    cmap=cmap, origin='lower', vmin=0, vmax=1)
    
    cbar = plt.colorbar()
    # cbar.set_label('Plausibility [-]')
    cbar.set_label('kumulative Evidenz [-]')
    
    # plt.xlabel('Frequency [\\si{\\hertz}]') 
    plt.xlabel('Frequenz [\\si{\\hertz}]')    
    if space_ind==2:
        # plt.ylabel('Top-TMD Accelerance $|\mathcal{H}_\mathrm{T-D}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
        plt.ylabel('Spitze-Dämpfer Akzeleranz $|\mathcal{H}_\mathrm{T-D}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
    else:
        # plt.ylabel('Top-Top Accelerance $|\mathcal{H}_\mathrm{T-T}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
        plt.ylabel('Spitze-Spitze Akzeleranz $|\mathcal{H}_\mathrm{T-T}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
    
    plt.subplots_adjust(top=0.97, bottom=0.14, left=0.095, right=1.0)
    
    # plt.savefig(f'figures/inc_avg_pl-frf-all.{space_ind}.png')
    # plt.savefig(f'figures/inc_avg_pl-frf-all.{space_ind}.pdf')
    plt.savefig(f'figures/inc_avg_pl-frf-all.{space_ind}.png')
    plt.savefig(f'figures/inc_avg_pl-frf-all.{space_ind}.pdf')
    plt.show()

In [ ]:
cm = 1/2.54 
plt.gcf().set_dpi(300)
plt.gcf().set_size_inches((2*4*cm,2*3*cm))
plt.gca().set_xticks([])
plt.gca().set_yticks([])
cbar.set_ticks([])
plt.ylabel('mittlere Akzeleranz')
plt.xlabel('Frequenz')
cbar.set_label('Plausibilität')
plt.subplots_adjust(top=0.99, bottom=0.1, left=0.08, right=1.0, wspace=0, hspace=0)
plt.savefig(f'polyuq_results/BB15-inc_avg_pl-frf-all.pdf')

In [ ]:
plt.close('all')
with matplotlib.rc_context(get_pcd('BB15')):
    fig, axes = plt.subplots(2,1,sharex=True, sharey=False)

    for space_ind in [1,2]:
        freq_ind = 206-1

        arr = np.load(os.path.join(result_dir,'estimations',f'frf-all.{space_ind}.npz'))
        all_focals_this, focals_mass = arr['all_focals_this'][:freq_ind,:], arr['focals_mass']

        all_focals_this *= 1000
        # omegas = np.arange(1,freq_ind+1)*5/1024*2*np.pi
        # all_focals_this /= omegas[:, np.newaxis, np.newaxis]**2
        # all_focals_this = np.log10(all_focals_this)
        
        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(all_focals_this,focals_mass,100)
        n_bins_bel = bel_stats.shape[1]

        cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
        im = axes[space_ind-1].imshow(pl_stats.T, aspect='auto', 
                                      extent= (0, 5/1025*freq_ind, np.min(all_focals_this), np.max(all_focals_this)),
                                      cmap=cmap, origin='lower', vmin=0, vmax=1)
        
        fig.supylabel('$|\mathcal{H}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$', y=0.6, fontsize=9)
        # if space_ind==2:
        #     axes[space_ind-1].set_ylabel('$|\mathcal{H}_\mathrm{T-D}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
        # else:
        #     axes[space_ind-1].set_ylabel('$|\mathcal{H}_\mathrm{T-T}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$')
    fig.align_ylabels()
    plt.subplots_adjust(top=0.97, bottom=0.12, left=0.095, right=1.0, hspace=0.1)
    cbar = fig.colorbar(im, ax=axes)
    # cbar.set_label('Plausibility [-]')
    cbar.set_label('kumulative Evidenz [-]')

    # axes[1].set_xlabel('Frequency [\\si{\\hertz}]')   
    axes[1].set_xlabel('Frequenz [\\si{\\hertz}]', labelpad=-9)
    axes[1].set_xticks([0,0.25,0.75,1.0])
    


#     plt.savefig(f'figures/inc_avg_pl-frf-low.all.png')
#     plt.savefig(f'figures/inc_avg_pl-frf-low.all.pdf')
    plt.savefig(f'figures/inc_avg_pl-frf-low.all.png')
    plt.savefig(f'figures/inc_avg_pl-frf-low.all.pdf')
    plt.show()

### Pure stochastic

In [ ]:
if False:
    poly_uq.load_state(samp_path, differential='samp')
    # hack around ice_mass polymorphy
    vars_epi[7]._focals[0][2] = 1
    poly_uq.weights_full_stoch()
    poly_uq.save_state(os.path.join(result_dir,'polyuq_samp_weights.npz'), differential='samp')
else:
    poly_uq.load_state(os.path.join(result_dir,'polyuq_samp_weights.npz'), differential='samp')
    
poly_uq.load_state(prop_path, differential='prop')
poly_uq.load_state(imp_path, differential='imp')

In [ ]:
def stoch_fun_ecdf(samp_flat, weight_flat, target_probabilities):    
    ecdf = np.cumsum(weight_flat)
    ecdf /= ecdf[-1]
    return np.interp(target_probabilities, ecdf, samp_flat)

def stoch_fun_hist(samp_flat, weight_flat, bins, density):
    hist, _ = np.histogram(samp_flat, bins, weights=weight_flat, density=density)
    return hist

ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
samp_path = os.path.join(result_dir,'polyuq_samp_weights.npz')
prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')

poly_uq.load_state(samp_path, differential='samp')
poly_uq.load_state(prop_path, differential='prop')
poly_uq.load_state(imp_path, differential='imp')

if True:
    if ret_name != 'frf':            
        # ensure same bins as for inc_avg_pl-ret_name-ret_ind
            
        inc_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_avg_inc.npz')
        poly_uq.load_state(inc_path, differential='inc')
        focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
        _, _, _, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
        
        # nbin_fact=20
        # n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
        # bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(poly_uq.N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2) # divide nbin_fact by 2 to account for reshaping intervals
        # stat_fun_hist.n_stat = len(bins_densities) - 1
        # cum_dens = False
        stat_fun_kwargs = {'bins':bins_bel, 'density':True}
    else:
        # ensure same bins as for inc_avg_pl-frf-xxx.x
        nbin_fact = 100
        n_hyc = len(poly_uq.hyc_mass(vars_epi))
        n_bins_dens = np.ceil(np.sqrt(n_hyc) * nbin_fact).astype(int)
        if space_ind==1:
            bins_densities = np.linspace(0, 0.012, n_bins_dens)
        elif space_ind==2:
            bins_densities = np.linspace(0, 0.0021, n_bins_dens)
        cum_dens = False
        stat_fun_kwargs = {'bins':bins_densities, 'density':True}

    stoch_path_part = 'polyuq_hist_stoch.npz'
    stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
    poly_uq.stat_full_stoch(stoch_fun_hist, stat_fun_kwargs)
    poly_uq.save_state(stoch_path, differential='stoch')
    
if True:
    n_stat = 100
    target_probabilities = np.linspace(0,1,n_stat)
    stat_fun_kwargs = {'target_probabilities':target_probabilities}
    stoch_path_part = 'polyuq_cdf_stoch.npz'
    stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
    poly_uq.stat_full_stoch(stoch_fun_ecdf, stat_fun_kwargs)
    poly_uq.save_state(stoch_path, differential='stoch')


In [ ]:
plt.figure()
stoch_path_part = 'polyuq_hist_stoch.npz'
stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
poly_uq.load_state(stoch_path, differential='stoch')
plt.step(bins_densities[:-1], poly_uq.stoch_stats[:,0,0])

plt.figure()
stoch_path_part = 'polyuq_cdf_stoch.npz'
stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
poly_uq.load_state(stoch_path, differential='stoch')
plt.step(poly_uq.stoch_stats[:,0,0], target_probabilities)

In [ ]:
with matplotlib.rc_context(get_pcd('print')):
    fig, axes = plt.subplots(1,3, sharey=True)
    ret_name='damp_freqs'
    colors=['dimgrey','grey']
    linestyles = ['solid', (0, (5, 1))]
    add_mode = 4 # 0 or 4
    if not add_mode:
        axes = np.insert(axes, 0, axes[0])
    for mode_pair,ax in enumerate(axes):
        for pair_member in range(2):
            mode_ind = (mode_pair + add_mode)* 2 + pair_member
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            
            samp_path = os.path.join(result_dir,'polyuq_samp.npz')
            prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
            imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
            
            stoch_path_part = 'polyuq_hist_stoch.npz'
            stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
            
            poly_uq.load_state(samp_path, differential='samp')
            poly_uq.load_state(prop_path, differential='prop')
            poly_uq.load_state(imp_path, differential='imp')
            poly_uq.load_state(stoch_path, differential='stoch')
            
            inc_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_avg_inc.npz')
            poly_uq.load_state(inc_path, differential='inc')
            focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
            _, _, _, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
            
            # nbin_fact=20
            # n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
            # bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(poly_uq.N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2) # divide nbin_fact by 2 to account for reshaping intervals

            # if not add_mode and mode_pair < 2:
            #     pl_stats /= 2

            ax.step(bins_bel[:-1], poly_uq.stoch_stats[:,0,0], where='post', color=colors[pair_member], linestyle=linestyles[pair_member])
            # ax.step(bins_bel, pl_stats[0,:], where='post',label='plausibility', color=colors[pair_member], linestyle=linestyles[pair_member])
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        
    custom_lines = [matplotlib.lines.Line2D([0], [0], color=colors[i], ls=linestyles[i]) for i in range(2)]
    fig.legend(custom_lines, ['$x$-$z$-plane', '$x$-$y$-plane',], loc=(0.78,0.84))

    # if not add_mode:
    #     axes[0].set_xlabel('$f^\\text{u,l}_{1} [\si{{\hertz}}]$')
    #     axes[2].set_xlabel('$f_{2} [\si{{\hertz}}]$')
    #     axes[3].set_xlabel('$f_{3} [\si{{\hertz}}]$')
    # else:
    #     axes[0].set_xlabel('$f_{4} [\si{{\hertz}}]$')
    #     axes[1].set_xlabel('$f_{5} [\si{{\hertz}}]$')
    #     axes[2].set_xlabel('$f_{6} [\si{{\hertz}}]$')
    if not add_mode:
        axes[0].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[2].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[3].set_xlabel('Frequency $[\si{{\hertz}}]$')
    else:
        axes[0].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[1].set_xlabel('Frequency $[\si{{\hertz}}]$')
        axes[2].set_xlabel('Frequency $[\si{{\hertz}}]$')


    axes[0].set_ylim(ymin=0)
    axes[0].set_ylabel('Probability Density [$\sfrac{1}{\si{\hertz}}$]')
    plt.subplots_adjust(top=0.97, bottom=0.135, left=0.105, right=0.97, wspace=0.12)
    if not add_mode:
        plt.savefig(f'figures/stoch_hist-damp_freqs-low.png')
        plt.savefig(f'figures/stoch_hist-damp_freqs-low.pdf')
    else:
        plt.savefig(f'figures/stoch_hist-damp_freqs-high.png')
        plt.savefig(f'figures/stoch_hist-damp_freqs-high.pdf')
    plt.show()
        

In [ ]:
with matplotlib.rc_context(get_pcd('print')):
    fig, ax = plt.subplots(1,1)
    ret_name='zetas'
    add_mode = 2
    # add_mode = 0
    if not add_mode:
        colors=['dimgrey','grey']
        labels=['$\zeta^\\text{l}_{1,y}$','$\zeta^\\text{l}_{1,z}$','$\zeta^\\text{u}_{1,y}$','$\zeta^\\text{u}_{1,z}$',]
    else:
        colors=['dimgrey','grey', 'lightgrey']
        labels=['$\zeta_{2,y}$','$\zeta_{2,z}$','$\zeta_{3,y}$','$\zeta_{3,z}$','$\zeta_{4,y}$','$\zeta_{4,z}$',]

    linestyles = ['solid', (0, (5, 1))]


    for mode_pair, color in enumerate(colors):
        for pair_member in range(2):
            mode_ind = (mode_pair  + add_mode) * 2 + pair_member
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            samp_path = os.path.join(result_dir,'polyuq_samp.npz')
            prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
            imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
            
            stoch_path_part = 'polyuq_hist_stoch.npz'
            stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
            
            poly_uq.load_state(samp_path, differential='samp')
            poly_uq.load_state(prop_path, differential='prop')
            poly_uq.load_state(imp_path, differential='imp')
            poly_uq.load_state(stoch_path, differential='stoch')
            
            inc_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_avg_inc.npz')
            poly_uq.load_state(inc_path, differential='inc')
            focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
            _, _, _, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
            # nbin_fact=20
            # n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
            # bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(poly_uq.N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2) # divide nbin_fact by 2 to account for reshaping intervals
            
            bins_bel *=100
            ax.step(bins_bel[:-1], poly_uq.stoch_stats[:,0,0], where='post', color=colors[pair_member], linestyle=linestyles[pair_member], label=labels[mode_pair*2+pair_member])

            # ax.step(bins_bel, pl_stats[0,:], where='post', color=color, linestyle=linestyles[pair_member], label=labels[mode_pair*2+pair_member])
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    # ax.set_xlabel(f'$f_{{{mode_pair+add_mode+1},y,z}} [\si{{\hertz}}]$')

    ax.set_ylim(ymin=0)
    if add_mode ==2:
        ax.set_xlim((0,2.4))
    else:
        ax.set_xlim((0,11))
        
    ax.set_ylabel('Probability Density [$\sfrac{1}{\si{\percent}}$]')
    ax.set_xlabel(f'Damping $\zeta [\si{{\percent}}]$')

    plt.legend(loc='upper right')
    plt.subplots_adjust(top=0.97, bottom=0.135, left=0.105, right=0.97, wspace=0.045)

    if not add_mode:
        plt.savefig(f'figures/stoch_hist-zetas-low.png')
        plt.savefig(f'figures/stoch_hist-zetas-low.pdf')
    else:
        plt.savefig(f'figures/stoch_hist-zetas-high.png')
        plt.savefig(f'figures/stoch_hist-zetas-high.pdf')
    plt.show()

In [ ]:
ret_name = 'frf'
all_stoch = []
space_ind = 2
for freq_ind in range(1025):
    ret_ind = {'frequencies':freq_ind, 'space':space_ind}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    
    stoch_path_part = 'polyuq_hist_stoch.npz'
    stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
    
    arr = np.load(stoch_path)
    stoch_stats = arr['self.stoch_stats'][:,0,0]
    all_stoch.append(stoch_stats)
    # print(focals_stats)
all_focals_this = np.stack(all_stoch)
np.savez(os.path.join(result_dir,'estimations',f'frf-stoch_all.{space_ind}.npz'), all_focals_this=all_focals_this)

In [ ]:
plt.close('all')
with matplotlib.rc_context(get_pcd('print')):
    fig, axes = plt.subplots(2,1,sharex=True, sharey=False)

    space_ind = 1
    freq_ind = 1024

    arr = np.load(os.path.join(result_dir,'estimations',f'frf-stoch_all.{space_ind}.npz'))
    all_focals_this = arr['all_focals_this'][:freq_ind,:]
    nbin_fact = 100
    n_hyc = len(poly_uq.hyc_mass(poly_uq.vars_epi))
    n_bins_dens = np.ceil(np.sqrt(n_hyc) * nbin_fact).astype(int)
    if space_ind==1:
        bins_densities = np.linspace(0, 0.012, n_bins_dens)
    elif space_ind==2:
        bins_densities = np.linspace(0, 0.0021, n_bins_dens)
    bins_densities *=1000
    
    cmap = matplotlib.cm.get_cmap('Greys', n_bins_dens)
    im = axes[1].imshow(all_focals_this.T, aspect='auto', 
                        extent= (0, 5/1025*freq_ind, np.min(bins_densities), np.max(bins_densities)),
                        cmap=cmap, origin='lower'#, norm='log'
                        , vmin=1e-4, vmax=500
                       )
    cbar = fig.colorbar(im, ax=axes[1])
    cbar.set_label('Prob. Density')
    
    
    
    arr = np.load(os.path.join(result_dir,'estimations',f'frf-all.{space_ind}.npz'))
    all_focals_this, focals_mass = arr['all_focals_this'][:freq_ind,:], arr['focals_mass']

    all_focals_this *= 1000
    bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(all_focals_this,focals_mass,100)
    n_bins_bel = bel_stats.shape[1]

    cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
    im = axes[0].imshow(pl_stats.T, aspect='auto', 
                                  extent= (0, 5/1025*freq_ind, np.min(all_focals_this), np.max(all_focals_this)),
                                  cmap=cmap, origin='lower', vmin=0, vmax=1)
    cbar = fig.colorbar(im, ax=axes[0])
    cbar.set_label('Plausibility')


    if space_ind==2:
        fig.supylabel('$|\mathcal{H}_\mathrm{T-D}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$',fontsize=10)
    else:
        fig.supylabel('$|\mathcal{H}_\mathrm{T-T}| [\\si{\\milli\\metre\\per\\second\\squared\\per\\newton}]$',fontsize=10)
        
    
    plt.subplots_adjust(top=0.97, bottom=0.125, left=0.12, right=1.0, hspace=0.1)
    

    axes[1].set_xlabel('Frequency [\\si{\\hertz}]')    


    plt.savefig(f'figures/inc_avg_pl-stoch_hist-frf-all.1.png')
    plt.savefig(f'figures/inc_avg_pl-stoch_hist-frf-all.1.pdf')
    plt.show()

In [ ]:
plt.subplots_adjust(top=0.97, bottom=0.125, left=0.12, right=1.0, hspace=0.1)

In [ ]:
plt.close('all')
ret_name = 'damp_freqs'
for mode in range(14):
    
    ret_ind = {'modes':mode}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
    samp_path = os.path.join(result_dir,'polyuq_samp.npz')
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
    imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
    
#     for inc_path_part in ['polyuq_hist_inc.npz', 'polyuq_cdf_inc.npz'][1:]:

    # inc_path_part = 'polyuq_hist_inc.npz'
    inc_path_part = 'polyuq_cdf_inc.npz'
    inc_path = os.path.join(result_dir, 'estimations', ret_dir, inc_path_part)
    stoch_path_part = 'polyuq_cdf_stoch.npz'
    stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
    poly_uq.load_state(samp_path, differential='samp')
    poly_uq.load_state(prop_path, differential='prop')
    poly_uq.load_state(imp_path, differential='imp')
    poly_uq.load_state(inc_path, differential='inc')
    poly_uq.load_state(stoch_path, differential='stoch')
    focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
    n_stat = focals_stats.shape[0]

    bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
    n_bins_bel = bins_bel.shape[0]
    with matplotlib.rc_context(get_pcd('print')):
        plt.figure()
        # fig, ax1 = plt.subplots()
        # plt.rc('text',usetex=True)
        # print(matplotlib.rcParams)

        n_stat = pl_stats.shape[0]
        target_probabilities = np.linspace(0,1,n_stat)
        if ret_name == 'zetas':
            bins_bel *= 100
        pcol = plt.pcolormesh(bins_bel, target_probabilities, pl_stats, cmap='Greys',rasterized=True)#, shading='gouraud')
        # pcol.set_edgecolor('none')
        n_stat = 100
        target_probabilities = np.linspace(0,1,n_stat)
        stoch_stats = poly_uq.stoch_stats[:,0,0]
        if ret_name == 'zetas':
            stoch_stats *= 100
        plt.plot(stoch_stats,target_probabilities, color='lightgrey')
        


        cbar = plt.colorbar()
        cbar.set_label('Plausibility [-]')
        if ret_name == 'zetas':
            plt.xlabel(f'Damping $\\zeta_{{{{{ret_ind["modes"]+1}}}}} [\si{{\percent}}_\\mathrm{{crit.}}]$')
            plt.ylabel('Cumulative Probability [$-$]')
        elif ret_name == 'damp_freqs':
            plt.xlabel(f'$f_{{{{{ret_ind["modes"] + 1}}}}} [\si{{\hertz}}]$')
            plt.ylabel('Cumulative Probability [$-$]')
        plt.ylim((0,1))
        plt.xlim((bins_bel.min(), bins_bel.max()))
        plt.subplots_adjust(top=0.975,bottom=0.125, left=0.09,right=1.0)
        plt.savefig(f'figures/inc_cdf_pl_stoch-{ret_dir}.png')
        plt.savefig(f'figures/inc_cdf_pl_stoch-{ret_dir}.pdf')
        #plt.show()

### Scatterplot Matrix

In [ ]:
'''
What do we need:
- we got the weights, they are the same for each output quantity
- check that weights are nicely distributed, if we use them for the alpha channel
- get some scatterplot matrix code
- output samples for all frequencies
for f1:
    get samples
    for f2:
        get samples
        if f1==f2:
            plot weighted histogram
        else:
            plot scatterplot matrix with weights to alpha channel

'''
import seaborn as sns
ret_name = 'damp_freqs'
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
samp_path = os.path.join(result_dir,'polyuq_samp_weights.npz')
poly_uq.load_state(samp_path, differential='samp')
data = {}
weights = poly_uq.stoch_weights
weights = weights.flatten()
non_zero = np.logical_and(weights!=0, weights<np.percentile(weights, 99.999))
data['weights'] = weights[non_zero]
data['weights'] /= data['weights'].max()
# plt.figure()
# plt.plot(data['weights'], data['weights'], ls='none',marker=',')
# plt.hist(data['weights'], bins=100)
# plt.show()

for mode_ind in range(14):
    ret_ind = {'modes':mode_ind}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
    poly_uq.load_state(prop_path, differential='prop')
    data[ret_dir] = poly_uq.out_samp.flatten()[non_zero]

data = pd.DataFrame(data=data)

In [ ]:
plt.close('all')
mode_start, mode_end = 0,6
# mode_start, mode_end = 6,14
num_modes = mode_end - mode_start

pcd = get_pcd('print')
pcd['ytick.right'] =  True 
pcd['ytick.labelright'] = True 
pcd['ytick.left'] = False
pcd['ytick.labelleft'] = False

with matplotlib.rc_context(pcd):

    fig, axes = plt.subplots(num_modes + 1,num_modes + 1, sharex='col', sharey='row')

    import matplotlib.colors
    color_s = matplotlib.colors.to_rgba('dimgrey', alpha=0)
    color_e = matplotlib.colors.to_rgba('dimgrey', alpha=1)
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('CustomCmap', colors=[color_s, color_e]) 
    clim = (np.infty,-np.infty)
    mappables = []
    for i,m1 in enumerate(range(mode_start, mode_end)):
        ax = axes[0,i+1]
        # ax = ax.twinx()
        ax.hist(data[f'{ret_name}-{m1}'],bins=100, weights=data['weights'], color='dimgrey')
        # ax.set_yticks([])
        for j,m2 in enumerate(reversed(range(mode_start, mode_end))):

            ax = axes[j+1,i+1]

            # if m1!=0:
            #     ax.set_yticks([])
            # if m2!=mode_ind:
            #     ax.set_xticks([])
            # if m2<m1: continue
            # continue
            # print(m1,m2)
            if m1==m2:
                continue
                # ax = ax.twinx()
                # ax.hist(data[f'{ret_name}-{m1}'],bins=100, weights=data['weights'], color='dimgrey')
                # ax.set_yticks([])
            else:
                xbins = 100
                ybins = 100
                #hist2d produces artifacts when using alpha colormaps due to internal use of pcolormesh (draws Patches)
                #ax.hist2d(x_, y_, bins=(xbins, ybins), density=True, cmap=cmap, norm=matplotlib.colors.LogNorm(), zorder=10)
                x_ = data[f'{ret_name}-{m1}']
                y_ = data[f'{ret_name}-{m2}']
                # compute the histogram and draw it as image
                counts, xedges, yedges=np.histogram2d(x_,y_,(xbins, ybins), density=True,  weights=data['weights'])
                clim = (min(clim[0],np.min(counts)),max(clim[1],np.max(counts)))
                mappable = ax.imshow(counts.T, cmap=cmap, norm=matplotlib.colors.PowerNorm(gamma=0.2),
                          zorder=10,
                           # interpolation='none',
                           origin='lower', resample=False, aspect='auto',
                           extent=(xedges.min(),xedges.max(), yedges.min(), yedges.max()))
                mappables.append(mappable)
    for j,m2 in enumerate(reversed(range(mode_start, mode_end))):
        ax = axes[j+1,0]
        # ax = ax.twinx()
        ax.hist(data[f'{ret_name}-{m2}'],bins=100, weights=data['weights'], color='dimgrey', orientation='horizontal')
    ax.invert_xaxis()
        # ax.set_yticks([])
    import matplotlib.ticker
    for ax in axes[-1,:]:
        ax.set_xticklabels(ax.get_xticks(), rotation = 90)
        if ax == axes[-1,0]:continue
        ax.xaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter('{x:1.3f}'))
    for m,ax in enumerate(axes[0,1:]):
        ax.xaxis.set_label_position('top')
        if ret_name=='damp_freqs':
            ax.set_xlabel(f'$f_{{{m+mode_start+1}}}$')
        else:
            ax.set_xlabel(f'$\zeta_{{{m+mode_start+1}}}$')
    for m,ax in enumerate(reversed(axes[1:,0])):
        ax.yaxis.set_label_position('left')
        if ret_name=='damp_freqs':
            ax.set_ylabel(f'$f_{{{m+mode_start+1}}}$')
        else:
            ax.set_ylabel(f'$\zeta_{{{m+mode_start+1}}}$')
    for ax in axes[1:,-1]:
        ax.yaxis.tick_right()
        ax.yaxis.set_ticks_position('right')
        ax.yaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter('{x:1.3f}'))

    for mappable in mappables:
        mappable.set_clim(clim)
    fig.subplots_adjust(hspace=0.14, wspace=0.12, top=0.95, left=0.05)
    plt.annotate('[\si{{\hertz}}]',  (0.92,0.04), xycoords='figure fraction')
    plt.savefig(f'figures/scatterplot-{ret_name}.{mode_start}-{mode_end}.png')
    plt.savefig(f'figures/scatterplot-{ret_name}.{mode_start}-{mode_end}.pdf')
    # fig.colorbar(mappable)

In [ ]:
text.remove()
with matplotlib.rc_context(pcd):
    text = plt.annotate('[\si{{\hertz}}]',  (0.92,0.04), xycoords='figure fraction')

In [ ]:
with matplotlib.rc_context(get_pcd('print')):
    fig, (ax1,ax2) = plt.subplots(2,1, sharex=True)
    ret_name='damp_freqs'
    colors=['dimgrey','grey']
    linestyles = ['solid', (0, (5, 1))]
    
    for mode_pair in range(2):
        for pair_member in range(2):
            mode_ind = mode_pair * 2 + pair_member
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            
            samp_path = os.path.join(result_dir,'polyuq_samp.npz')
            prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
            imp_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
            
            stoch_path_part = 'polyuq_hist_stoch.npz'
            stoch_path = os.path.join(result_dir, 'estimations', ret_dir, stoch_path_part)
            
            poly_uq.load_state(samp_path, differential='samp')
            poly_uq.load_state(prop_path, differential='prop')
            poly_uq.load_state(imp_path, differential='imp')
            poly_uq.load_state(stoch_path, differential='stoch')
            
            inc_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_avg_inc.npz')
            poly_uq.load_state(inc_path, differential='inc')
            focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
            bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
            # print(scipy.integrate.trapezoid(poly_uq.stoch_stats[:,0,0],bins_bel[:-1]))
            ax2.step(bins_bel[:-1], poly_uq.stoch_stats[:,0,0], where='post', color=colors[pair_member], linestyle=linestyles[pair_member])
            # print(np.sum(pl_stats))
            dbins = np.diff(bins_bel)
            # pl_stats[0,1:] /= dbins
            if mode_pair < 2:
                pl_stats /= 2
            bins_bel = np.append(bins_bel, bins_bel[-1:], axis=0)
            pl_stats = np.append(pl_stats, [[0]], axis=1)
            bins_bel = np.insert(bins_bel, 0,bins_bel[:1], axis=0)
            pl_stats = np.insert(pl_stats, 0,[[0]], axis=1)
            
            # print(scipy.integrate.trapezoid(pl_stats,bins_bel))
            ax1.step(bins_bel, pl_stats[0,:], where='post',label='plausibility', color=colors[pair_member], linestyle=linestyles[pair_member])
            # ax.step(bins_bel, pl_stats[0,:], where='post',label='plausibility', color=colors[pair_member], linestyle=linestyles[pair_member])
        ax2.xaxis.set_major_locator(plt.MaxNLocator(9))
        
    custom_lines = [matplotlib.lines.Line2D([0], [0], color=colors[i], ls=linestyles[i]) for i in range(2)]
    leg = fig.legend(custom_lines, ['$x$-$z$-plane', '$x$-$y$-plane',], loc=(0.78,0.41))

    # ax1.set_xlabel('Frequency $[\si{{\hertz}}]$')
    ax2.set_xlabel('Frequency $[\si{{\hertz}}]$')


    ax1.set_ylim((0,0.65))
    ax1.set_ylabel('Mass [-]')
    # ax1.yaxis.set_ticks_position('left')
    ax2.set_ylim(ymin=0)
    # ax2.yaxis.set_label_position('right')
    # ax2.yaxis.set_ticks_position('right')
    ax2.set_ylabel('Probability Density [$\sfrac{1}{\si{\hertz}}$]')
    plt.subplots_adjust(top=0.97, bottom=0.135, left=0.105, right=0.97, hspace=0.12)
    fig.align_ylabels()
    plt.savefig(f'figures/inc_avg_pl-stoch_hist-damp_freqs.first_order.png')
    plt.savefig(f'figures/inc_avg_pl-stoch_hist-damp_freqs.first_order.pdf')
    
    
    plt.show()
        

## Todo


- ✔️ check other out_quants (higher frequencies, damping ratios, etc.)
- ✔️ find good epsilon for zetas and damp_freqs
- ✔️ evaluate intp_errors, exceed and undershot for frf, zetas, damp_freqs
- ✔️ check most recent ale samples manually to see what's going on

- ✔️ also do something similar for nearestND interpolator
- ✔️increase number of propositions for add_mass and b and ???
    - ✔️damp_freqs is sensitive to add_mass (1 proposition), ice_mass (1 proposition), (b 1 and t 2)
    - ✔️zetas is sensititive to zeta (and dD but that is aleatory) (4 propositions)
    - ✔️frfs is sensitive to all of them depending on frequency line and response dof
    - ✔️-> should have two propositions on add_mass (40,50)
- ✔️ estimate runtime for all 13717 aleatory samples with nearest and rbf interpolators and decide 
  - ✔️ whether performance can be improved further
  - ✔️ whether to implement distributed imperfection estimation
  - ✔️ 1 sample around 30 s -> 3062 samples -> 1531 minutes -> 26 hours (114 hours -> 5 days)
  - ✔️ Vectorize wrapper: 1 sample around 12 s @ 4 CPUs (rbf) -> 3062 samples -> 612 (2743) minutes -> 10:30 (45) hours (2 days) ->  (2078 out_quants -> 659 days @ 1 node, 32 days @ 20 nodes)

- ✔️ test val_samp_prim
- batch process already propagated ale samples

- for exemplary ret_names do
  - estimate_imp (with rbf)
  - optimize_inc
  - draw plots
- test repeat with nearest and compare
- repeat for all ret_names and selected ret_inds
- finish remaining ale samples

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
%load_ext snakeviz

import sys
import os
import logging
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import scipy.stats.qmc

import ray

from polyuq import PolyUQ
from polyuq.data_manager import DataManager

%aimport -sys -logging -matplotlib -matplotlib.pyplot -numpy -pandas - scipy.stats -scipy.stats.qmc

In [ ]:
from UQ_Modal_FEM import mapping_function, mapping_pass, test_interpolation, vars_definition, est_imp, opt_inc, est_stoch

##### Uncertainty Modeling

In [ ]:
vars_ale, vars_epi, arg_vars = vars_definition()

##### Sampling

In [ ]:
dim_ex = 'cartesian'

# %%snakeviz
N_mcs_ale = 13717 # N_mcs = 1e6 = N_mcs_ale * N_mcs_epi
N_mcs_epi = 729 # = 3^6 = 2.56^n_imp ~ 3^n_imp -> cover every corner and midpoints in a full-factorial design (but distributed)
use_dm = True
result_dir = 'polyuq_results/'

poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

poly_uq.load_state(os.path.join(result_dir,'polyuq_samp.npz'))

## Evaluate RBF interpolation parameters

In [ ]:
ret_name = ['damp_freqs','zetas','frf'][1]
if ret_name == 'frf':
    ret_ind = {'frequencies':126, 'space':1}
else:
    ret_ind = {'modes':13}
    
ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'        
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

poly_uq.load_state(os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz'))
if ret_name == 'damp_freqs':
    poly_uq.out_valid = [np.nanmin(poly_uq.out_samp),np.nanmax(poly_uq.out_samp)]

In [ ]:
plt.close('all')
# %%snakeviz -t
if True:
    logger= logging.getLogger('polyuq.polymorphic_uncertainty')
    logger.setLevel(level=logging.DEBUG)
    poly_uq.N_mcs_ale=np.random.randint(20,3062)
    poly_uq.estimate_imp(
        interp_fun='rbf',
        opt_meth='genetic',
        plot_res=False,
        plot_intp=False,
        intp_err_warn = 0,
        extrp_warn = 0,
        start_ale = poly_uq.N_mcs_ale - 2,
        kernel='gaussian',
        epsilon={'frf':4,'zetas':2,'damp_freqs':2}[ret_name]
    )
    print([var for var in poly_uq.vars_ale if var.primary] + list(poly_uq.vars_imp))
    for n_ale in range(poly_uq.N_mcs_ale - 2,poly_uq.N_mcs_ale):
        for i_hyc in range(len(poly_uq.imp_hyc_foc_inds)):
            print(n_ale, i_hyc)
            print(poly_uq.val_samp_prim[n_ale,i_hyc,:,0])
            print(poly_uq.val_samp_prim[n_ale,i_hyc,:,1])

#     poly_uq.save_state(os.path.join(result_dir,'estimations', f'{ret_dir}/polyuq_imp.npz'))
else:
    poly_uq.load_state(os.path.join(result_dir,'estimations', f'{ret_dir}/polyuq_imp.npz'))
    

In [ ]:
# intp_previous4 = np.copy(poly_uq.intp_errors)
# exceed_previous4 = np.copy(poly_uq.intp_exceed)
# undershot_previous4 = np.copy(poly_uq.intp_undershot)
plt.figure()
plt.hist(100-poly_uq.intp_errors*100, bins=np.arange(101))
# plt.hist(100-intp_previous1*100, bins=np.arange(101))
# plt.hist(100-intp_previous2*100, bins=np.arange(101))
plt.xlabel("Interpolator Accuracy %")
plt.xlim((0,100))
plt.show()
# print(poly_uq.intp_exceed[1] / poly_uq.intp_exceed[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)
# if poly_uq.intp_undershot[0]:
#     print(poly_uq.intp_undershot[1] / poly_uq.intp_undershot[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)
# print(poly_uq.imp_foc)
# print(exceed_previous1[1] / exceed_previous1[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)
# print(undershot_previous1[1] / undershot_previous1[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)
# print(exceed_previous2[1] / exceed_previous2[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)
# print(undershot_previous2[1] / undershot_previous2[0] / (poly_uq.out_valid[1] - poly_uq.out_valid[0]) * 100)

In [ ]:
ray.shutdown()
ray.init(address='auto')

all_intp_errors = {'damp_freqs':[None for _ in range(30)],'zetas':[None for _ in range(30)],'frf':[None for _ in range(30)]}
all_intp_exceed = {'damp_freqs':[None for _ in range(30)],'zetas':[None for _ in range(30)],'frf':[None for _ in range(30)]}
all_intp_undershot = {'damp_freqs':[None for _ in range(30)],'zetas':[None for _ in range(30)],'frf':[None for _ in range(30)]}

In [ ]:


# rets = test_interpolation('damp_freqs',{'modes':11},754)
# print(rets)

# asffdafgdf


@ray.remote
def sub_test(i, ret_name, i_n, N_mcs):
    if ret_name == 'frf':
        ret_ind = {'frequencies':i_n//2, 'space':i_n%2}
    else:
        ret_ind = {'modes':i_n}
    return i, ret_name, i_n, *test_interpolation(ret_name, ret_ind, N_mcs)

all_inds ={'damp_freqs':[],'zetas':[],'frf':[]}
all_MCS ={'damp_freqs':[],'zetas':[],'frf':[]}
futures = []
for ret_name in ['damp_freqs','zetas','frf']:
    if ret_name == 'frf':
        n = 126*2
    else:
        n=14
        
    inds = np.random.randint(0,n,30)
    N_mcs = np.random.randint(100,3062,30)
    all_inds[ret_name] = inds
    all_MCS[ret_name] = N_mcs
    for i in range(30):
        if all_intp_errors[ret_name][i] is not None: continue
        futures.append(sub_test.remote(i, ret_name, inds[i], N_mcs[i]))

futures = set(futures)

In [ ]:
while True:
    ready, wait = ray.wait(
        list(futures), num_returns=min(len(futures), 10), timeout=30)
    try:
        ret_sets = ray.get(ready)
        size_before = len(futures)
        for i, ret_name, ind, intp_errors, intp_exceed, intp_undershot in ret_sets:
        #     assert all_intp_errors[ret_name][i] is None
            all_intp_errors[ret_name][i] = intp_errors
        #     assert all_intp_exceed[ret_name][i] is None
            all_intp_exceed[ret_name][i] = intp_exceed
        #     assert all_intp_undershot[ret_name][i] is None
            all_intp_undershot[ret_name][i] = intp_undershot
            
        futures.difference_update(ready)
        print(f"Finished {len(ready)} samples. Remaining {len(futures)} samples. (before {size_before})")
    except Exception as e:
        print(e)
        pass
    if len(futures) == 0:
        break

In [ ]:
plt.close('all')
for ret_name in ['damp_freqs','zetas','frf']:
    '''
    draw 30 histograms of intp_errors (overlaying transparent)
    draw histogram of average exceed / undershot percent
    '''
    this_intp_errors = all_intp_errors[ret_name]
    plt.figure()
    for intp_errors in this_intp_errors:
        if intp_errors is None: continue
        plt.hist(100-intp_errors*100, bins=np.arange(101), alpha=0.1)
        
    plt.title(ret_name + ' interpolator error')
    plt.savefig('nearest ' + ret_name + ' interpolator error')
    continue
    all_undershots = []
    num_undershots = []
    for undershot in all_intp_undershot[ret_name]:
        if undershot is None: continue
        all_undershots.append(undershot[1] / undershot[0])
        num_undershots.append(undershot[0])
    plt.figure()
    plt.hist(all_undershots, weights= num_undershots)
    plt.title(ret_name + ' undershot error')
    plt.savefig('nearest ' + ret_name + ' undershot error')
#     plt.figure()
#     plt.hist(num_undershots)
#     plt.title(ret_name + ' undershot numbers')
    all_exceeds = []
    num_exceeds = []
    for exceed in all_intp_exceed[ret_name]:
        if exceed is None: continue
        all_exceeds.append(exceed[1] / exceed[0])
        num_exceeds.append(exceed[0])
    plt.figure()
    plt.hist(all_exceeds, weights= num_exceeds)
    plt.title(ret_name + ' exceed error')
    plt.savefig('nearest ' + ret_name + ' exceed error')
#     plt.figure()
#     plt.hist(num_exceeds)
#     plt.title(ret_name + ' exceed numbers')
    

### variability and incompleteness

In [ ]:
def stat_fun(a, weight,i_stat):
    return np.average(a, weights=weight)
n_stat = 1

# def stat_fun(a, weight, i_stat):
#     n = len(a)
#     mean = np.average(a, weights=weight)
#     std = np.sqrt(np.cov(a, aweights=weight))
#     if std==0:
#         conf= [mean, mean]
#     else:
#         sem = std / np.sqrt(n)
#         conf = scipy.stats.t.interval(alpha=0.95, df=n-1, loc=mean, scale=sem) 
#     if i_stat is not None:
#         return conf[i_stat]
#     else: 
#         return conf

# n_stat = 2
# focals_stats, hyc_mass = poly_uq.estimate_inc(stat_fun, n_stat)
focals_stats, hyc_mass = poly_uq.optimize_inc(stat_fun, n_stat)
# poly_uq.save_state(os.path.join(result_dir,f'{ret_dir}/polyuq_avg_inc.npz'))

bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, hyc_mass, 10, False)
n_bins_bel = bins_bel.shape[0]

In [ ]:
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
for ret_name in ['damp_freqs','zetas','frf']:
    if ret_name == 'frf':
        continue
        #frequencies = np.arange(2048)
        all_focals = []
        space_ind = 3
        for freq_ind in range(1025):
            ret_ind = {'frequencies':freq_ind, 'space':space_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            arr = np.load(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz'))
#             poly_uq.load_state(f'polyuq_results/polyuq_avg_inc.npz')
            focals_stats, focals_mass = arr['self.focals_stats'], arr['self.focals_mass']
            all_focals.append(focals_stats)
            
        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(np.stack([foc[0,:,:] for foc in all_focals]),focals_mass)
        n_bins_bel = bel_stats.shape[1]
        cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
        plt.figure()
        im = plt.imshow(pl_stats.T, aspect='auto', extent= (0, 5, np.min(focals_stats), np.max(focals_stats)),
                        cmap=cmap, origin='lower')
        
        
    else:
#         continue
#         fig, (ax1,ax2) = plt.subplots(1,2,gridspec_kw={'width_ratios': [3, 1]}, sharey=True)
        fig, ax1 = plt.subplots()
            
        for mode_ind in range(14):
            ret_ind = {'modes':mode_ind}
            ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
            
#             poly_uq.load_state(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_lci_inc.npz'))
#             focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
#             if False:
#                 plot_focals(focals_stats[0,:,:], focals_mass, ax2)
#             else:
#                 bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 1, False)
#                 ax2.step(bins_bel, pl_stats[0], where='post',label='plausibility ')
            
            poly_uq.load_state(os.path.join(result_dir,'estimations',f'{ret_dir}/polyuq_avg_inc.npz'))
            focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
            
            if False:
                plot_focals(focals_stats[0,:,:], focals_mass, ax1)
            else:
                bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
                ax1.step(bins_bel, pl_stats[0], where='post',label='plausibility ')




            
# plt.step(bins_bel, bel_stats[0], where='post',label='belief')
# plt.step(bins_bel, bel_stats[1], where='post',label='belief')
# plt.step(bins_bel, pl_stats[0], where='post',label='plausibility ')
# plt.step(bins_bel, pl_stats[1], where='post', c='grey')
# plt.step(bins_bel, pl_stats[0], where='post', c='grey')
# plt.fill_between(bins_bel, pl_stats[0], pl_stats[1], step='post',label='plausibility', color='lightgrey')
# plt.step(bins_bel, q_stats[0], where='post',label='commonality')
# plt.legend()
# plt.xlabel(f"{ret_dir}")
# plt.ylabel("mass")
# None

In [ ]:
all_focals_this = np.stack([foc[0,:,:] for foc in all_focals])
# all_focals_this /= np.max(all_focals_this)
all_focals_this[all_focals_this<=1e-3] = 1e-3
all_focals_this[all_focals_this>80] = 80
# all_focals_this = 10*np.log10(all_focals_this)
bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(all_focals_this,focals_mass,1)
np.savez('polyuq_results/frf_pl.npz', all_focals_this, bel_stats, pl_stats, q_stats, bins_bel)
n_bins_bel = bel_stats.shape[1]
cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
with matplotlib.rc_context(print_context_dict2):
    plt.figure()
    print(np.min(all_focals_this), np.max(all_focals_this))
    im = plt.imshow(pl_stats.T, aspect='auto', extent= (0, 200, np.min(all_focals_this), np.max(all_focals_this)),
                    cmap=cmap, origin='lower', vmin=0, vmax=1)
    plt.xlabel('Frequency [\\si{\\hertz}]')
    plt.ylabel('FRF Magnitude [\\si{\\metre\\per\\second\\squared}]')

In [ ]:
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

if 0:
    fig, (ax1,ax2) = plt.subplots(1,2,gridspec_kw={'width_ratios': [3, 1]}, sharey=True)
    ret_name='damp_freqs'
    ret_ind = {'modes':0}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_lci_inc.npz'))
    focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
    plot_focals(focals_stats[0,:,:], focals_mass, ax2)

    poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_avg_inc.npz'))
    focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass
    plot_focals(focals_stats[0,:,:], focals_mass, ax1)
    
if 1:
    fig, axes = plt.subplots(1,3, sharey=True)
    ret_name='damp_freqs'
    for mode_ind,ax in enumerate(axes):
        ret_ind = {'modes':mode_ind}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
        poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_avg_inc.npz'))
        focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass

        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
        ax.step(bins_bel, pl_stats[0], where='post',label='plausibility')

if 1:
    fig, ax = plt.subplots(1,1, sharey=True)
    ret_name='zetas'
    for mode_ind in range(3):
        ret_ind = {'modes':mode_ind}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
        poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_avg_inc.npz'))
        focals_stats, focals_mass = poly_uq.focals_stats, poly_uq.focals_mass

        bel_stats, pl_stats, q_stats, bins_bel = aggregate_mass(focals_stats, focals_mass, 10, False)
        ax.step(bins_bel, pl_stats[0], where='post',label='plausibility')

In [ ]:
# ret_name = 'frf'
# space_ind = 9
# freq_ind = 100
# ret_ind = {'frequencies':freq_ind, 'space':space_ind}
ret_name = 'damp_freqs'
# ret_name = 'zetas'
mode_ind = 0
ret_ind = {'modes':mode_ind}

ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

if True:
    poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_imp.npz'))
    
    def stat_fun(a, weight, i_stat, bins_densities, cum_dens):
        '2. Quantify Variability for each incomplete sample and imprecise hypercube'
        if i_stat is None:
            hist,_ = np.histogram(a, bins_densities, weights=weight, density=True)

            if cum_dens:
                hist= np.cumsum(hist)
                hist /= hist[-1]   
            return hist
        else:
            # factor 6 faster but more error prone for "dirty" data, uneven bins or ....
            if cum_dens: first_edge = bins_densities[0]
            else: first_edge = bins_densities[i_stat]
            last_edge = bins_densities[i_stat + 1]
            keep  = (a >= first_edge)
            keep &= (a <=  last_edge)
            return np.sum(weight[keep])


    nbin_fact=20
    n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
    bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2) # divide nbin_fact by 2 to account for reshaping intervals
    n_stat = len(bins_densities) - 1
    cum_dens = False
    stat_fun_kwargs = {'bins_densities':bins_densities, 'cum_dens':cum_dens}#, 'ax':ax1}

    # focals_densities, hyc_mass = poly_uq.estimate_inc(intervals, stat_fun, n_stat, stat_fun_kwargs)
    focals_stats, focals_mass = poly_uq.optimize_inc(stat_fun, n_stat, stat_fun_kwargs)
    # poly_uq.save_state(os.path.join(result_dir,f'{ret_dir}/polyuq_hist_inc.npz'))
else:
    poly_uq.load_state(os.path.join(result_dir,f'{ret_dir}/polyuq_hist_inc.npz'))
    
    nbin_fact=20
    n_imp_hyc = len(poly_uq.imp_hyc_foc_inds)
    bins_densities = generate_histogram_bins(poly_uq.imp_foc.reshape(N_mcs_ale, n_imp_hyc * 2), 1, nbin_fact/2) # divide nbin_fact by 2 to account for reshaping intervals
    cum_dens = True
    
    focals_stats = poly_uq.focals_stats
    focals_mass = poly_uq.focals_mass

In [ ]:
focals_stats = focals_densities
focals_mass = hyc_mass
bel_densities, pl_densities, q_densities, bins_bel = aggregate_mass(focals_stats, focals_mass, 20, False)
n_bins_bel = bel_densities.shape[1]

In [ ]:

bpq=2

fig, (ax1, ax2) = plt.subplots(2,1,sharex=True,sharey=True)   

for i_hyc in range(hyc_mass.shape[0]):
    l2d=ax1.step(bins_densities[:-1], focals_stats[:,i_hyc, 0], where='post')
    c = l2d[0].get_color()
    ax1.step(bins_densities[:-1], focals_stats[:,i_hyc, 1], where='post', color=c)

if bpq == 1:
    belief = bel_densities.T
    label = 'Belief [-]'
elif bpq == 2:
    belief = pl_densities.T
    label = 'Plausibility [-]'
elif bpq == 3:
    belief = q_densities.T
    label = 'Commonality [-]'

cmap = matplotlib.cm.get_cmap('Greys', n_bins_bel)
im = ax2.imshow(belief, aspect='auto', extent= (bins_densities[0], bins_densities[-1], 0, 1), 
           cmap=cmap, origin='lower', vmin=0, vmax=1)

cbar = fig.colorbar(im, ax=[ax1,ax2])
cbar.set_label(label)

ax2.grid(True, alpha=0.5)
ax2.set_xlabel("Output quantity [-]")

if cum_dens:
    ax2.set_ylabel("Cumulative Probability Density [-]")
else:
    ax2.set_ylabel("Probability Density [-]")    

## distributed quantification (ray) ###

In [ ]:
ray.shutdown()

In [ ]:
ray.init(address='auto')
futures = []
remote_opt_inc = ray.remote(num_cpus=1)(opt_inc)
remote_est_imp = ray.remote(num_cpus=4)(est_imp)
remote_est_stoch = ray.remote(num_cpus=1)(est_stoch)

In [ ]:
poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)

logger= logging.getLogger('polyuq.polymorphic_uncertainty')
logger.setLevel(level=logging.INFO)

def submit_imp():
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    fname = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_imp.npz')
    if not os.path.exists(fname) or True:
        futures.append(remote_est_imp.remote(poly_uq, result_dir, ret_name, ret_ind))
    else:
        
        intp_err = np.load(fname)['self.intp_errors']
        nan_ind = np.where(np.isnan(intp_err))[0]
        if not len(nan_ind):
            print(f'{ret_dir}/polyuq_imp.npz already finished.')
            return # finished
        elif np.min(nan_ind)>=5390: # should not be the case
            print(f'{ret_dir}/polyuq_imp.npz already finished.')
            return # finished
        else:
            futures.append(remote_est_imp.remote(poly_uq, result_dir, ret_name, ret_ind))
        

def submit_inc():
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    fname = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_avg_inc.npz')
    if not os.path.exists(fname) or True:
        futures.append(remote_opt_inc.remote(poly_uq, result_dir, ret_name, ret_ind))
    else:
        print(f'{ret_dir}/polyuq_avg_inc.npz already finished.')
        
def submit_stoch():
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    fname = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_hist_stoch.npz')
    if not os.path.exists(fname) or True:
        futures.append(remote_est_stoch.remote(poly_uq, result_dir, ret_name, ret_ind))
    else:
        print(f'{ret_dir}/polyuq_hist_stoch.npz already finished.')

   
    
def submit():
    # submit_imp()
    # submit_inc()
    submit_stoch()
    
if False:
    futures = list(futures)
else:
    futures = []


for ret_name in ['damp_freqs','zetas','frf']:
    if ret_name != 'frf':
        # continue
        for mode_ind in range(14):
            ret_ind = {'modes':mode_ind}
            # est_stoch(poly_uq, result_dir, ret_name, ret_ind)
            submit()
    else:
        # continue
        for freq_ind in range(1025):
            for space_ind in [2,1]:
                ret_ind = {'frequencies':freq_ind, 'space':space_ind}
                submit()
futures = set(futures)

In [ ]:
while True:
    ready, wait = ray.wait(
        list(futures), num_returns=min(len(futures), 10), timeout=60)
    finished = []
    failed = []
    for obj_ref in ready:
        try:
            ret_sets = ray.get(obj_ref)
            finished.append(obj_ref)
        except ray.exceptions.RayTaskError as e:
            print(e)
            failed.append(obj_ref)
    
    size_before = len(futures)
    futures.difference_update(finished)
    futures.difference_update(failed)
    print(f"Finished {len(finished)}, failed {len(failed)} samples. Remaining {len(futures)} samples. (before {size_before})")
        
    if len(futures) == 0:
        break

### Evidential Probability Density Estimation

In [ ]:
m=15
s=2
size=717

plt.close('all')
plt.figure()
norm = scipy.stats.norm(m,s)
x = np.linspace(norm.ppf(0.001),scipy.stats.norm(2*m,2*s).ppf(0.999), 100)

plt.plot(x, norm.pdf(x)/2+scipy.stats.norm(2*m,2*s).pdf(x)/2, 'r-')

samp = np.concatenate([norm.rvs(size//2),scipy.stats.norm(2*m,2*s).rvs(size//2+size%2)])

samp = np.sort(samp)

# kernel density estimation
# kde = scipy.stats.gaussian_kde(samp)
# plt.plot(x, kde(x), 'b-')

#histogram
plt.hist(samp, bins='fd',density=True, cumulative=False, alpha=0.5)

# sliding bin histogram
ntot = len(samp)
bin_width=(samp[-1]-samp[0])*0.05
bin_width = np.diff(np.quantile(samp,[0.25,0.75]))/np.power(ntot,1/3)*2
bin_centers, densities = np.zeros(len(samp)), np.zeros(len(samp))
for n in range(ntot):
    ind = np.logical_and(samp > (samp[n] - bin_width / 2), samp < (samp[n] + bin_width / 2))
    p = np.sum(ind) / bin_width / ntot
    bin_centers[n] = samp[n]
    densities[n] = p
# plt.plot(bin_centers, densities, ls='none',marker=',')

target_pdfs = np.linspace(0,1,101)
intervals = np.zeros((target_pdfs.shape[0],2))

for j in range(target_pdfs.shape[0]):
    target_pdf = target_pdfs[j]
    for k,min_max in enumerate([-1,1]):# maximum: -1, minimum: 1
        it = range(len(densities))
        if min_max==-1:
            it = reversed(it)
        for i in it:
            if densities[i]>target_pdf:
                break
        else:
            intervals[j,k] = np.nan
            continue
        # intervals[j,k] = bin_centers[i]
        # continue
        if i <= 1:
            intervals[j,k] = bin_centers[i]
        elif i >= densities.shape[0] - 1:
            intervals[j,k] = bin_centers[-1]
        else:
            if min_max==-1:
                intervals[j,k] = (bin_centers[i]*(densities[i+1] - target_pdf) + bin_centers[i+1]*(target_pdf - densities[i]))/(densities[i+1] - densities[i])
            else:
                intervals[j,k] = (bin_centers[i - 1]*(densities[i] - target_pdf) + bin_centers[i]*(target_pdf - densities[i-1]))/(densities[i] - densities[i-1])

cm = 0
for j in range(target_pdfs.shape[0]):
    r,l = intervals[j,:]
    m = target_pdfs[1]-target_pdfs[0]
    plt.bar(l,m,(r-l),bottom = cm+0.05*m, align='edge', color='lightgrey', edgecolor='black', alpha=0.5)
    cm += m
   

In [ ]:
def weighted_quantile(values, quantiles, sample_weight=None, 
                      values_sorted=False, old_style=False):
    """ Very close to numpy.percentile, but supports weights.
    NOTE: quantiles should be in [0, 1]!
    :param values: numpy.array with data
    :param quantiles: array-like with many quantiles needed
    :param sample_weight: array-like of the same length as `array`
    :param values_sorted: bool, if True, then will avoid sorting of
        initial array
    :param old_style: if True, will correct output to be consistent
        with numpy.percentile.
    :return: numpy.array with computed quantiles.
    """
    values = np.array(values)
    quantiles = np.array(quantiles)
    if sample_weight is None:
        sample_weight = np.ones(len(values))
    sample_weight = np.array(sample_weight)
    assert np.all(quantiles >= 0) and np.all(quantiles <= 1), \
        'quantiles should be in [0, 1]'

    if not values_sorted:
        sorter = np.argsort(values)
        values = values[sorter]
        sample_weight = sample_weight[sorter]

    weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
    if old_style:
        # To be convenient with numpy.percentile
        weighted_quantiles -= weighted_quantiles[0]
        weighted_quantiles /= weighted_quantiles[-1]
    else:
        weighted_quantiles /= np.sum(sample_weight)
    return np.interp(quantiles, weighted_quantiles, values)


m1=15
s1=2
m2=2*m1
s2=2*s1
size=13717


norm1 = scipy.stats.norm(m1,s1)
norm2 = scipy.stats.norm(m2,s2)

import scipy.stats.qmc
supp1 = [norm1.ppf(0.001), norm1.ppf(0.999)]
supp2 = [norm2.ppf(0.001), norm2.ppf(0.999)]

engine = scipy.stats.qmc.Halton(2)
samp = engine.random(size) # (size,2)
# samp = scipy.stats.qmc.scale(samp,(supp1[0],supp2[0]),(supp1[1],supp2[1]))
samp = scipy.stats.qmc.scale(samp,(supp1[0],supp1[0]),(supp2[1],supp2[1]))

pdf1 = norm1.pdf(samp[:,0])
pdf2 = norm2.pdf(samp[:,0])
pdf = pdf1 + pdf2
pdf /= np.sum(pdf)

# x = np.concatenate((samp[:,0], samp[:,1]))
samp = samp[:,0] #+ samp[:,1]
# x = np.random.choice(samp.flat, samp.shape[0])

def density(samp, weights, i_stat, min_max, target_densities, bin_width=None):
    ind = np.argsort(samp)
    samp = samp[ind]
    weights = weights[ind]
    
    if min_max == -1:
        samp = np.flip(samp)
        weights = np.flip(weights)
        
    N_mcs_ale = len(samp)
    target_p = target_densities[i_stat]
    if bin_width is None: # should probably be fixed prior to optimize_inc. how?
        quantiles = weighted_quantile(samp, [0.25,0.75], weights,True)
        bin_width = np.diff(quantiles)/np.power(len(N_mcs_ale),1/3) * 2 # fd rule
    p_prev = 0
    it = range(N_mcs_ale)
    # if min_max==-1:
    #     it = reversed(it)
    for n in it:
        '''
        looking for a contiguous sequence of weights, that is >= target_p * bin_width
        maybe there is another way of finding that sequence
        two variables determine p
        the density of samp in a given sequence and the values of weights
        if diff(samp) is small, we have a high density
        
        we want to find a stopping criterion, to stop early, if no density with requested value is present in the dataset
        problem is, there can always be an accumulation of samples at any point in the sequence which leads to a high density
        
        pre-computation by vectorization will lead to and ind matrix of size 13717x13717 -> also not an option
        bisection or random searches might also just miss the single bin, that has the highest density
        we have to extend optimize_inc such that we can select a range of i_stat to perform optimzation on and extend that later, 
        if we realize, we are missing the peaks of the distribution
        the optimizer will also fail, if wrapper returns 0 all the time, that might be an option for breaking the loop early
        '''
        ind = np.logical_and(samp > (samp[n] - bin_width / 2), samp < (samp[n] + bin_width / 2))
        p_curr = np.sum(weights[ind]) / bin_width #/ ntot
        if p_curr > target_p:
            break
        p_prev = p
    else: # did not find a bin with target density
        return np.nan
    # return samp[n]
    if n == 0:
        return samp[n]
    # if n == N_mcs_ale - 1:
    #     return samp[n]
    
    # interpolate between two bins
    # elif min_max==-1: # walking backwards -> interpolating from the end
    #     x0, x1 = p_prev, p_curr
    #     y0, y1 = samp[n + 1], samp[n]
    # else:
    x0, x1 = p_prev, p_curr
    y0, y1 = samp[n - 1], samp[n]
        
    return (y0 * (x1 - target_pdf) + y1 * (target_pdf - x0))/(x1 - x0)
         

ind = np.argsort(samp)
samp = samp[ind]
pdf = pdf[ind]
plt.close('all')
plt.figure()
# plt.plot(samp, pdf, ls='none', marker=',')
plt.hist(samp, 
         weights=pdf, 
         density=True,
        bins=50)
pass

In [ ]:
plt.close('all')
plt.figure()
# plt.plot(samp, pdf, ls='none', marker=',')
plt.hist(samp, 
         weights=pdf, 
         density=True,
        bins=50)
# plt.hist(norm1.rvs(size)+norm2.rvs(size), bins=50, alpha=0.5, density=True)
# sliding bin histogram

target_pdfs = np.linspace(0,0.2,50)
intervals = np.full((target_pdfs.shape[0],2),np.nan)

ntot = len(samp)
quantiles = weighted_quantile(samp, [0.25,0.75], pdf,True)
bin_width = np.diff(quantiles)/np.power(ntot,1/3)*2

for i_stat in range(target_pdfs.shape[0]):
    print(i_stat)
    for k, min_max in enumerate([-1,1]):
        intervals[i_stat,k] = density(samp, pdf, i_stat, min_max, target_pdfs, bin_width)
    if np.isnan(intervals[i_stat,:]).all():
        break

                
cm = 0
for j in range(target_pdfs.shape[0]):
    r,l = intervals[j,:]
    m = target_pdfs[1]-target_pdfs[0]
    plt.bar(l,m,(r-l),bottom = cm+0.05*m, align='edge', color='lightgrey', edgecolor='black', alpha=0.5)
    cm += m

In [ ]:
# plt.figure()
# plt.plot(samp, pdf, ls='none', marker=',')
plt.hist(samp, 
         weights=pdf, 
         density=True,
        bins=50, 
        alpha=0.5)
# plt.hist(norm1.rvs(size)+norm2.rvs(size), bins=50, alpha=0.5, density=True)
# sliding bin histogram

bin_centers, densities = np.zeros(len(samp)), np.zeros(len(samp))
for n in range(ntot):
    ind = np.logical_and(samp > (samp[n] - bin_width / 2), samp < (samp[n] + bin_width / 2))
    p = np.sum(pdf[ind]) / bin_width #/ ntot
    bin_centers[n] = samp[n]
    densities[n] = p
# plt.plot(bin_centers, densities, ls='none',marker=',')

for j in range(target_pdfs.shape[0]):
    target_pdf = target_pdfs[j]
    for k,min_max in enumerate([-1,1]):# maximum: -1, minimum: 1
        it = range(len(densities))
        if min_max==-1:
            it = reversed(it)
        for i in it:
            if densities[i]>target_pdf:
                break
        else:
            intervals[j,k] = np.nan
            continue
            
        # intervals[j,k] = bin_centers[i]
        # continue
        if i <= 1:
            intervals[j,k] = bin_centers[i]
        elif i >= densities.shape[0] - 1:
            intervals[j,k] = bin_centers[-1]
        else:
            if min_max==-1:
                intervals[j,k] = (bin_centers[i]*(densities[i+1] - target_pdf) + bin_centers[i+1]*(target_pdf - densities[i]))/(densities[i+1] - densities[i])
            else:
                intervals[j,k] = (bin_centers[i - 1]*(densities[i] - target_pdf) + bin_centers[i]*(target_pdf - densities[i-1]))/(densities[i] - densities[i-1])


                
cm = 0
for j in range(target_pdfs.shape[0]):
    r,l = intervals[j,:]
    m = target_pdfs[1]-target_pdfs[0]
    plt.bar(l,m,(r-l),bottom = cm+0.05*m, align='edge', color='lightgrey', edgecolor='black', alpha=0.5)
    cm += m